In [1]:
# gemma-2-2b_batch_top_k_width-2pow14_date-0107/resid_post_layer_12/trainer_9 -> BatchTopK, k=160
# gemma-2-2b_batch_top_k_width-2pow14_date-0107/resid_post_layer_12/trainer_8 -> BatchTopK, k=80
# context is 1024

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# This works on an A40 (48GB VRAM); may need to adjust for other hardware
EVAL_BATCH_SIZE = 8
TOKENIZER_BATCH_SIZE = 256
DEVICE = "cuda:0"

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        base_model,
        {},
        num_layers=base_model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=base_model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [2]:
from transformers_sae.ops import find_latest_checkpoint, load_checkpoint

eval_layer = 16
saes = {}
for sae_spec in ("next_layer",):
    checkpoint_dir = f"/workspace/sae_checkpoints/gemma_2_2b/{sae_spec}"
    checkpoint = find_latest_checkpoint(checkpoint_dir, eval_layer)
    saes[sae_spec] = load_checkpoint(checkpoint).sae

In [10]:
from transformers_sae.sae_bench import run_sae_bench_evals

eval_results = run_sae_bench_evals(
    model,
    tokenizer,
    saes["next_layer"],
    eval_layer,
    batch_size=1,
    # batch_size=EVAL_BATCH_SIZE,
    num_tokens=int(1e3),
    dataset=validation_dataset.shuffle(seed=42),
)
print(eval_results[0])



Reconstruction Batches: 100%|███████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.38s/it]


Sparsity and Variance Batches: 100%|████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.02s/it]


{'model_behavior_preservation': {'kl_div_score': 0.991838907747062, 'kl_div_with_ablation': 11.509650230407715, 'kl_div_with_sae': 0.09393131732940674}, 'model_performance_preservation': {'ce_loss_score': 0.9925627694959213, 'ce_loss_with_ablation': 12.403467178344727, 'ce_loss_with_sae': 1.2156283855438232, 'ce_loss_without_sae': 1.1317983865737915}, 'reconstruction_quality': {'explained_variance': 0.8520269393920898, 'explained_variance_legacy': 0.7729664444923401, 'mse': 6.940868377685547, 'cossim': 0.920372724533081}, 'shrinkage': {'l2_norm_in': 215.66696166992188, 'l2_norm_out': 199.8099822998047, 'l2_ratio': 0.9252700805664062, 'relative_reconstruction_bias': 1.005124807357788}, 'sparsity': {'l0': 82.2869873046875, 'l1': 739.8115844726562}, 'token_stats': {'total_tokens_eval_reconstruction': 1024, 'total_tokens_eval_sparsity_variance': 1024}}


In [9]:
from transformers_sae.validation import run_validations
import numpy as np

validations = run_validations(
    model,
    tokenizer,
    {eval_layer: saes["next_layer"]},
    validation_dataset.shuffle(seed=42),
    TOKENIZER_BATCH_SIZE,
    1,
    # EVAL_BATCH_SIZE,
    int(1e3),
    start_layer=eval_layer,
    # end_layer=eval_layer + 1,
    eval_layers=[eval_layer, model.num_layers]
)
print("L0", np.mean(validations.layer_results[eval_layer].l0))
print("KL", np.mean(validations.layer_results[model.num_layers].kl))

Running SAE evals:   0%|          | 0/1000 [00:00<?, ?it/s]

L0 82.28699
KL 0.09393132


In [5]:
from transformers_sae.tokenization import make_dataloader
from transformers_sae.metrics import kl_eval

with (
    torch.no_grad(),
    torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True),
):
    batch = next(
        make_dataloader(
            model, tokenizer, validation_dataset.shuffle(seed=42), None, 256, 1
        ).__iter__()
    )
    batch.to(model.device)
    saes["next_layer"].onload()
    logits_1 = base_model(
        batch.input_ids,
        attention_mask=batch.attention_mask,
        position_ids=batch.position_ids,
        use_cache=False,
    )[0]
    logits_2 = make_replacement_model(model, {eval_layer: saes["next_layer"]})(
        batch.input_ids,
        attention_mask=batch.attention_mask,
        position_ids=batch.position_ids,
        use_cache=False,
        token_mask=batch.token_mask,
        pass_through_positions=batch.special_token_indices,
    )[0]
    print(
        torch.nn.functional.kl_div(
            logits_2.log_softmax(-1),
            logits_1.log_softmax(-1),
            log_target=True,
            reduction="none",
        ).sum(dim=-1)[batch.token_mask.bool()].mean().item()
    )
    print(np.mean(kl_eval(logits_2.log_softmax(-1), logits_1.log_softmax(-1), batch, return_type="np")))
    saes["next_layer"].offload()

0.09393131732940674
0.09393132
